## Визуализация аугментаций (Speed classifier)

Этот ноутбук показывает **3 случайных примера** из `train`-сплита:
- слева: исходное изображение из датасета
- справа: то, что реально подаётся в модель после аугментаций из `training/speed_classifier/dataset.py`

Запусти кодовую ячейку несколько раз — аугментации будут меняться (они случайные).


In [16]:
import os
import random
import sys
import torch
import torchvision
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

In [ ]:
# Чтобы импорт `training.speed_classifier.*` работал и из ноутбука
# В Jupyter текущая папка может быть не корнем проекта, поэтому ищем вверх по дереву.
ROOT = Path.cwd().resolve()
for _ in range(8):
    if (ROOT / "training" / "speed_classifier" / "dataset.py").is_file():
        break
    if ROOT.parent == ROOT:
        break
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    from training.speed_classifier.dataset import DatasetConfig, load_split_dataset  # noqa: E402
except ModuleNotFoundError:
    # fallback: если ноутбук запущен из training/speed_classifier
    from dataset import DatasetConfig, load_split_dataset  # type: ignore

# --- ПАРАМЕТРЫ (можно править) ---
DATA_DIR = "datasets/speed_cls_v1"
IMAGE_SIZE = 128
N_SAMPLES = 3
FIG_W = 12
FIG_H_PER_ROW = 4

# Какой чекпоинт оценивать
CKPT_PATH = ROOT / "training/speed_classifier/runs/run1/best.pt"

# Если хочешь воспроизводимые картинки, задай seed (или поставь None)
SEED = None

# Matplotlib иногда пытается писать кеш в домашнюю директорию
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "training/speed_classifier/.mplconfig"))

'/Users/elmanovsm/personal/cv/graduation/training/speed_classifier/.mplconfig'

In [ ]:
if SEED is not None:
    random.seed(int(SEED))
    np.random.seed(int(SEED))
    torch.manual_seed(int(SEED))

# DATA_DIR делаем абсолютным от корня проекта
data_dir_abs = (ROOT / DATA_DIR).resolve()
if not (data_dir_abs / "train").is_dir():
    raise FileNotFoundError(
        f"Не найден датасет: {data_dir_abs}/train\n"
        f"Текущий ROOT: {ROOT}\n"
        f"Подсказка: сначала запусти make_dataset/generate_speed_classifier_dataset.py"
    )

cfg = DatasetConfig(data_dir=str(data_dir_abs), image_size=int(IMAGE_SIZE))
ds = load_split_dataset(cfg, "train")

mean = torch.tensor(cfg.mean)[:, None, None]
std = torch.tensor(cfg.std)[:, None, None]


def denorm(x: torch.Tensor) -> torch.Tensor:
    # x: CHW, normalized
    y = x.detach().cpu() * std + mean
    return torch.clamp(y, 0.0, 1.0)


def to_hwc_uint8(x01_chw: torch.Tensor) -> np.ndarray:
    arr = (x01_chw.permute(1, 2, 0).numpy() * 255.0).round().astype(np.uint8)
    return arr


# Выбираем случайные индексы
idxs = [random.randrange(0, len(ds)) for _ in range(int(N_SAMPLES))]

fig, axes = plt.subplots(nrows=int(N_SAMPLES), ncols=2, figsize=(FIG_W, FIG_H_PER_ROW * int(N_SAMPLES)))
if int(N_SAMPLES) == 1:
    axes = np.array([axes])

for row, idx in enumerate(idxs):
    path, y = ds.samples[idx]
    class_name = ds.classes[int(y)]

    # Оригинал
    orig = Image.open(path).convert("RGB")
    axes[row, 0].imshow(orig)
    axes[row, 0].set_title(f"ORIG | {class_name}\n{Path(path).as_posix()}")
    axes[row, 0].axis("off")

    # Аугментированная версия (то, что реально идёт в модель)
    x_aug, _ = ds[idx]  # здесь применяются аугментации
    x_vis = to_hwc_uint8(denorm(x_aug))
    axes[row, 1].imshow(x_vis)
    axes[row, 1].set_title(f"AUG (train transforms) | {class_name}")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()
